# Machine Learning for Biology

**Tier 3 -- Applied Bioinformatics**

Machine learning (ML) has become indispensable in modern bioinformatics: classifying protein functions, predicting drug responses, identifying regulatory elements, and analyzing single-cell data. This notebook introduces the core ML concepts and provides hands-on experience with scikit-learn on biological data.

**Prerequisites:** Tier 2 (Python, NumPy, pandas), statistics basics  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`

---

[← Previous: Statistics for Bioinformatics](../../Tier_3_Applied_Bioinformatics/06_Statistics_for_Bioinformatics/01_statistics_for_bioinformatics.ipynb) | [Next: From Sequence to Discovery: An Integrative Bioinformatics Project →](../../Tier_3_Applied_Bioinformatics/08_Capstone_Project/01_capstone_project.ipynb)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 1. Machine Learning in Bioinformatics: Use Cases

| Application | ML type | Input features | Output |
|-------------|---------|---------------|--------|
| Variant pathogenicity (CADD, REVEL) | Classification | Conservation, protein impact, allele frequency | Pathogenic / Benign |
| Gene expression prediction | Regression | Promoter sequence, histone marks | Expression level |
| Protein function prediction | Classification | Sequence features, domains, GO terms | Functional category |
| Drug response | Classification/Regression | Gene expression, mutations | Responder / IC50 |
| Cell type identification (scRNA-seq) | Clustering | Gene expression profiles | Cell type labels |
| Promoter recognition | Classification | k-mer frequencies, CpG content, TATA box | Promoter / Non-promoter |
| Protein structure (AlphaFold) | Deep learning | Sequence + MSA + templates | 3D coordinates |

---
## 2. Supervised Learning: Classification vs Regression

### Classification
The output is a **discrete category**: tumor vs. normal, protein family A vs. B vs. C.

### Regression
The output is a **continuous value**: gene expression level, drug IC50, protein stability.

Both follow the same workflow:
1. **Collect data** with known labels (training set)
2. **Engineer features** that capture relevant information
3. **Train a model** to map features to labels
4. **Evaluate** on held-out data
5. **Predict** on new, unlabeled data

---
## 3. Feature Engineering for Biological Data

Raw biological data (DNA sequences, protein structures) cannot be fed directly into most ML algorithms. We need to convert them into **numerical feature vectors**.

### 3.1 k-mer frequencies

For DNA/protein sequences, count the frequency of all subsequences of length k.

- **k=1**: nucleotide composition (4 features for DNA)
- **k=2**: dinucleotide frequencies (16 features for DNA)
- **k=3**: trinucleotide / codon frequencies (64 features for DNA)

In [ ]:
from itertools import product

def kmer_frequencies(sequence, k=2):
    """Compute normalized k-mer frequencies for a DNA sequence."""
    sequence = sequence.upper()
    # Generate all possible k-mers
    all_kmers = [''.join(p) for p in product('ACGT', repeat=k)]
    counts = {km: 0 for km in all_kmers}
    
    total = 0
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in counts:
            counts[kmer] += 1
            total += 1
    
    # Normalize to frequencies
    if total > 0:
        counts = {km: c / total for km, c in counts.items()}
    
    return counts

# Example
example_seq = 'ATCGATCGCGATCGATATCGCGCGATATATCGATCG'
freqs = kmer_frequencies(example_seq, k=2)
print("Dinucleotide frequencies:")
for km, f in sorted(freqs.items()):
    if f > 0:
        print(f"  {km}: {f:.3f}", end='  ')
print()

### 3.2 Physicochemical properties

For protein sequences, features can include:
- Amino acid composition (20 features)
- Molecular weight, isoelectric point
- Hydrophobicity profile
- Secondary structure propensities

### 3.3 Sequence-derived features for promoters

Drawing from the rice promoter dataset used in this course:
- GC content in different windows
- CpG observed/expected ratio
- TATA box presence/absence
- TFBS density and types
- Methylation levels
- SNP density

In [ ]:
def extract_promoter_features(sequence, tss_pos=None):
    """Extract a feature vector from a promoter sequence."""
    seq = sequence.upper()
    length = len(seq)
    if tss_pos is None:
        tss_pos = length // 2
    
    features = {}
    
    # Overall composition
    features['gc_content'] = (seq.count('G') + seq.count('C')) / length
    features['at_content'] = 1 - features['gc_content']
    
    # k-mer features (dinucleotides)
    di_freqs = kmer_frequencies(seq, k=2)
    for km, f in di_freqs.items():
        features[f'di_{km}'] = f
    
    # CpG observed/expected
    n_c = seq.count('C')
    n_g = seq.count('G')
    n_cpg = seq.count('CG')
    features['cpg_oe'] = (n_cpg * length) / (n_c * n_g) if (n_c > 0 and n_g > 0) else 0
    
    # TATA box presence
    upstream = seq[max(0, tss_pos-50):tss_pos]
    features['has_tata'] = 1 if 'TATAAA' in upstream else 0
    
    # GC content in upstream vs downstream windows
    up_500 = seq[max(0, tss_pos-500):tss_pos]
    down_500 = seq[tss_pos:min(length, tss_pos+500)]
    features['gc_upstream'] = (up_500.count('G') + up_500.count('C')) / max(len(up_500), 1)
    features['gc_downstream'] = (down_500.count('G') + down_500.count('C')) / max(len(down_500), 1)
    features['gc_ratio'] = features['gc_upstream'] / max(features['gc_downstream'], 0.01)
    
    return features

# Test on a random sequence
test_seq = ''.join(np.random.choice(list('ACGT'), 2000))
feats = extract_promoter_features(test_seq)
print(f"Number of features: {len(feats)}")
print("\nSample features:")
for k, v in list(feats.items())[:10]:
    print(f"  {k}: {v:.4f}")

---
## 4. Building a Biological Dataset: Promoter Classification

Let us create a realistic dataset for classifying DNA sequences as **promoter** or **non-promoter**. This is inspired by the rice promoter analysis project.

Promoter sequences will have:
- Higher GC content near the center
- Higher CpG density
- Occasional TATA boxes

Non-promoter sequences will have:
- Random genomic background composition

In [ ]:
def generate_promoter(length=500):
    """Generate a synthetic promoter sequence with biologically realistic features."""
    seq = []
    center = length // 2
    for i in range(length):
        dist = abs(i - center)
        gc_bias = 0.3 + 0.25 * np.exp(-0.5 * (dist / (length * 0.15)) ** 2)
        p = [0.5 * (1 - gc_bias), gc_bias / 2, gc_bias / 2, 0.5 * (1 - gc_bias)]
        seq.append(np.random.choice(list('ACGT'), p=p))
    
    # Add CpG enrichment
    for i in range(center - 100, center + 100):
        if i < length - 1 and np.random.random() < 0.15:
            seq[i] = 'C'
            seq[i + 1] = 'G'
    
    # Add TATA box with 30% probability
    if np.random.random() < 0.3:
        tata_pos = center - np.random.randint(25, 35)
        if tata_pos > 0 and tata_pos + 7 < length:
            for j, b in enumerate('TATAAAG'):
                seq[tata_pos + j] = b
    
    return ''.join(seq)

def generate_non_promoter(length=500):
    """Generate a random genomic (non-promoter) sequence."""
    weights = [0.29, 0.21, 0.21, 0.29]  # typical AT-rich vertebrate genome
    return ''.join(np.random.choice(list('ACGT'), size=length, p=weights))

# Generate dataset
np.random.seed(42)
n_pos = 300  # promoters
n_neg = 300  # non-promoters

sequences = []
labels = []

for _ in range(n_pos):
    sequences.append(generate_promoter())
    labels.append(1)

for _ in range(n_neg):
    sequences.append(generate_non_promoter())
    labels.append(0)

labels = np.array(labels)
print(f"Dataset: {len(sequences)} sequences")
print(f"  Promoters: {(labels == 1).sum()}")
print(f"  Non-promoters: {(labels == 0).sum()}")

In [ ]:
# Extract features for all sequences
feature_list = []
for seq in sequences:
    feature_list.append(extract_promoter_features(seq))

features_df = pd.DataFrame(feature_list)
features_df['label'] = labels

print(f"Feature matrix shape: {features_df.shape}")
print(f"\nFeature means by class:")
print(features_df.groupby('label')[['gc_content', 'cpg_oe', 'has_tata', 'di_CG']].mean().round(4))

In [ ]:
# Visualize feature distributions by class
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

plot_features = ['gc_content', 'cpg_oe', 'di_CG', 'gc_upstream']
plot_titles = ['GC Content', 'CpG Observed/Expected', 'CG Dinucleotide Freq', 'Upstream GC Content']

for ax, feat, title in zip(axes.flat, plot_features, plot_titles):
    prom_vals = features_df[features_df['label'] == 1][feat]
    non_vals = features_df[features_df['label'] == 0][feat]
    ax.hist(non_vals, bins=30, alpha=0.6, color='gray', label='Non-promoter', density=True)
    ax.hist(prom_vals, bins=30, alpha=0.6, color='steelblue', label='Promoter', density=True)
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Feature Distributions: Promoter vs Non-Promoter', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Train/Test Split and Cross-Validation

### Why we split data

A model must be evaluated on data it has **never seen during training**. Otherwise, we cannot distinguish genuine patterns from memorized noise (overfitting).

### Cross-validation

Instead of a single split, **k-fold cross-validation** rotates through k different train/test splits and averages the results. This gives a more reliable estimate of model performance.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# Prepare feature matrix and labels
feature_cols = [c for c in features_df.columns if c != 'label']
X = features_df[feature_cols].values
y = features_df['label'].values

# 80/20 train/test split (stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples ({(y_train == 1).sum()} promoters, {(y_train == 0).sum()} non-promoters)")
print(f"Test set:     {X_test.shape[0]} samples ({(y_test == 1).sum()} promoters, {(y_test == 0).sum()} non-promoters)")
print(f"Features:     {X_train.shape[1]}")

---
## 6. Key Algorithms

### 6.1 Logistic Regression

Despite the name, logistic regression is a **classification** algorithm. It models the probability of class membership using the logistic function:

$$P(y=1|x) = \frac{1}{1 + e^{-(w^T x + b)}}$$

**Strengths:** Interpretable coefficients, fast, works well with many features.  
**Weaknesses:** Assumes linear decision boundary in feature space.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Feature scaling (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Non-promoter', 'Promoter']))

In [ ]:
# Feature importance from logistic regression coefficients
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_n = 15
top_feats = coef_df.head(top_n)
colors = ['steelblue' if c > 0 else 'coral' for c in top_feats['coefficient']]
ax.barh(range(top_n), top_feats['coefficient'], color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feats['feature'])
ax.set_xlabel('Coefficient (positive = promoter-associated)')
ax.set_title('Top 15 Features -- Logistic Regression')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 6.2 Random Forest

An **ensemble** of decision trees, each trained on a random subset of data and features. The final prediction is the majority vote.

**Strengths:** Handles non-linear relationships, robust to outliers, provides feature importance, minimal tuning needed.  
**Weaknesses:** Less interpretable than logistic regression, can overfit on very small datasets.

The source material for this course used Random Forest to classify wine varieties using chemical properties, achieving ~97% accuracy -- demonstrating the power of this approach.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)  # RF does not require feature scaling

y_pred_rf = rf.predict(X_test)
print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Non-promoter', 'Promoter']))

In [ ]:
# Feature importance from Random Forest
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_n = 15
top_rf = rf_importance.head(top_n)
ax.barh(range(top_n), top_rf['importance'], color='forestgreen')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_rf['feature'])
ax.set_xlabel('Feature importance (Gini)')
ax.set_title('Top 15 Features -- Random Forest')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 6.3 Support Vector Machine (SVM)

SVM finds the hyperplane that maximally separates classes. With the **kernel trick**, it can handle non-linear boundaries.

**Strengths:** Effective in high-dimensional spaces, robust to overfitting with proper regularization.  
**Weaknesses:** Slow on large datasets, requires feature scaling, less interpretable.

In [ ]:
from sklearn.svm import SVC

# Train SVM with RBF kernel
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
print("=== SVM (RBF kernel) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.3f}")
print()
print(classification_report(y_test, y_pred_svm, target_names=['Non-promoter', 'Promoter']))

### 6.4 k-Nearest Neighbors (k-NN)

Classifies a sample based on the majority class among its k nearest neighbors in feature space.

**Strengths:** Simple, no training phase, adapts to complex boundaries.  
**Weaknesses:** Slow at prediction time for large datasets, sensitive to irrelevant features and the curse of dimensionality.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
print("=== k-Nearest Neighbors (k=7) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.3f}")
print()
print(classification_report(y_test, y_pred_knn, target_names=['Non-promoter', 'Promoter']))

---
## 7. Model Evaluation

### 7.1 Metrics overview

| Metric | Formula | What it measures |
|--------|---------|------------------|
| **Accuracy** | (TP + TN) / Total | Overall correctness |
| **Precision** | TP / (TP + FP) | Of predicted positives, how many are correct? |
| **Recall (Sensitivity)** | TP / (TP + FN) | Of actual positives, how many did we find? |
| **F1 score** | 2 * Precision * Recall / (P + R) | Harmonic mean of precision and recall |
| **ROC AUC** | Area under ROC curve | Discrimination ability across all thresholds |

**When to use which:**
- Balanced classes: accuracy is fine
- Imbalanced classes (e.g., rare variants): focus on precision/recall/F1
- Comparing models: ROC AUC is threshold-independent

In [ ]:
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                             precision_recall_curve, f1_score)

# Confusion matrices for all models
models = {
    'Logistic Regression': (y_pred_lr, lr.predict_proba(X_test_scaled)[:, 1]),
    'Random Forest': (y_pred_rf, rf.predict_proba(X_test)[:, 1]),
    'SVM': (y_pred_svm, svm.predict_proba(X_test_scaled)[:, 1]),
    'k-NN': (y_pred_knn, knn.predict_proba(X_test_scaled)[:, 1]),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (y_pred, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Non-prom', 'Promoter'])
    ax.set_yticklabels(['Non-prom', 'Promoter'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f"{name}\nAcc={accuracy_score(y_test, y_pred):.3f}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=16, color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['steelblue', 'forestgreen', 'coral', 'mediumpurple']

for (name, (_, y_prob)), color in zip(models.items(), colors):
    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={roc_auc:.3f})")
    
    # Precision-Recall curve
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ax2.plot(rec, prec, color=color, linewidth=2, label=name)

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves')
ax1.legend(loc='lower right')

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

### 7.2 Cross-validation comparison

In [ ]:
from sklearn.pipeline import Pipeline

# Define pipelines (including scaling where needed)
pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', random_state=42))
    ]),
    'k-NN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':<25} {'Mean Accuracy':>14} {'Std':>8}")
print("-" * 50)
cv_results = {}
for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:<25} {scores.mean():>14.3f} {scores.std():>8.3f}")

In [ ]:
# Visualize cross-validation results
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot([cv_results[name] for name in pipelines],
                labels=list(pipelines.keys()), widths=0.5)
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 7.3 Overfitting demonstration

In [ ]:
# Show overfitting with increasing tree depth
depths = range(1, 30)
train_scores = []
test_scores = []

for d in depths:
    rf_d = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42)
    rf_d.fit(X_train, y_train)
    train_scores.append(rf_d.score(X_train, y_train))
    test_scores.append(rf_d.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(depths), train_scores, 'o-', color='steelblue', label='Training accuracy')
ax.plot(list(depths), test_scores, 'o-', color='coral', label='Test accuracy')
ax.set_xlabel('Max tree depth')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Training vs Test Accuracy\n(gap = overfitting)')
ax.legend()
ax.set_ylim(0.85, 1.02)
plt.tight_layout()
plt.show()

print("As max_depth increases, training accuracy reaches 1.0 (memorization)")
print("but test accuracy levels off or decreases. The gap indicates overfitting.")

---
## 8. Unsupervised Learning

When we have no labels, unsupervised methods discover structure in the data.

### 8.1 Dimensionality Reduction

High-dimensional biological data (e.g., expression of 20,000 genes) needs to be reduced for visualization and to combat the curse of dimensionality.

| Method | Type | Preserves | Use case |
|--------|------|-----------|----------|
| **PCA** | Linear | Global variance | Initial exploration, feature reduction |
| **t-SNE** | Non-linear | Local neighborhoods | Visualization of clusters |
| **UMAP** | Non-linear | Both local and global | scRNA-seq visualization |

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Simulate a gene expression dataset with 3 cell types
np.random.seed(42)
n_cells = 300
n_genes = 100

# Three cell types with different expression profiles
cell_type_a = np.random.normal(5, 1.5, (100, n_genes))
cell_type_a[:, :20] += 3  # upregulated genes 0-19

cell_type_b = np.random.normal(5, 1.5, (100, n_genes))
cell_type_b[:, 20:40] += 4  # upregulated genes 20-39

cell_type_c = np.random.normal(5, 1.5, (100, n_genes))
cell_type_c[:, 40:60] += 3.5  # upregulated genes 40-59

X_expr = np.vstack([cell_type_a, cell_type_b, cell_type_c])
cell_labels = np.array(['T cell'] * 100 + ['B cell'] * 100 + ['Macrophage'] * 100)

print(f"Expression matrix: {X_expr.shape} (cells x genes)")

In [ ]:
# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_expr)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_expr)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_map = {'T cell': 'steelblue', 'B cell': 'coral', 'Macrophage': 'forestgreen'}
point_colors = [colors_map[c] for c in cell_labels]

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=point_colors, s=15, alpha=0.7)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('PCA')

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=point_colors, s=15, alpha=0.7)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title('t-SNE')

# Legend
for label, color in colors_map.items():
    axes[0].scatter([], [], c=color, label=label)
    axes[1].scatter([], [], c=color, label=label)
axes[0].legend()
axes[1].legend()

plt.suptitle('Dimensionality Reduction of Gene Expression Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PCA variance explained
pca_full = PCA(n_components=20)
pca_full.fit(X_expr)

fig, ax = plt.subplots(figsize=(10, 5))
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
ax.bar(range(1, 21), pca_full.explained_variance_ratio_, color='steelblue', alpha=0.7, label='Individual')
ax.plot(range(1, 21), cumvar, 'ro-', label='Cumulative')
ax.axhline(0.9, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Proportion of Variance Explained')
ax.set_title('PCA Scree Plot')
ax.legend()
plt.tight_layout()
plt.show()

n_90 = np.argmax(cumvar >= 0.9) + 1
print(f"Components needed for 90% variance: {n_90}")

### 8.2 Clustering: k-means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Cluster the expression data
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_expr)

# Evaluate clustering quality
ari = adjusted_rand_score(cell_labels, cluster_labels)
sil = silhouette_score(X_expr, cluster_labels)

print(f"Adjusted Rand Index: {ari:.3f} (1.0 = perfect match with true labels)")
print(f"Silhouette Score: {sil:.3f} (higher = better separation)")

# Cross-tabulation
ct = pd.crosstab(cell_labels, cluster_labels, rownames=['True'], colnames=['Cluster'])
print(f"\nCross-tabulation (true labels vs clusters):")
print(ct)

In [ ]:
# Elbow method to find optimal k
inertias = []
sil_scores = []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_expr)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_expr, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(list(k_range), inertias, 'bo-')
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia (within-cluster sum of squares)')
ax1.set_title('Elbow Method')

ax2.plot(list(k_range), sil_scores, 'ro-')
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette score')
ax2.set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()

print(f"Optimal k by silhouette: {list(k_range)[np.argmax(sil_scores)]}")

---
## 9. Deep Learning: Brief Introduction

Deep learning has revolutionized bioinformatics in recent years:

| Application | Architecture | Example tool |
|-------------|-------------|---------------|
| Protein structure prediction | Attention / Transformer | **AlphaFold** |
| Regulatory sequence analysis | 1D CNN | **DeepSEA**, **Basset** |
| Variant effect prediction | CNN + attention | **Enformer**, **SpliceAI** |
| Drug-target interaction | Graph neural networks | **DeepDTA** |
| Single-cell analysis | Autoencoders | **scVI** |

### CNN for DNA sequences (concept)

A **1D Convolutional Neural Network** slides learned filters (like PWMs) across a one-hot encoded DNA sequence. The key insight: CNN filters learn motifs directly from data without manual feature engineering.

```
Input: One-hot encoded DNA (4 x L matrix)
  -> Conv1D filters (learn motifs like TATA, CpG patterns)
  -> Pooling (aggregate over positions)
  -> Dense layers (combine features)
  -> Output (classification/regression)
```

We will not implement deep learning here (it requires PyTorch/TensorFlow and GPU resources), but understanding the concept is important.

In [ ]:
# Demonstrate one-hot encoding for DNA sequences (input to CNNs)
def one_hot_encode_dna(sequence):
    """Convert a DNA sequence to a one-hot encoded matrix (4 x length)."""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = np.zeros((4, len(sequence)))
    for i, base in enumerate(sequence.upper()):
        if base in mapping:
            encoded[mapping[base], i] = 1
    return encoded

# Show example
example = 'ATCGATCG'
encoded = one_hot_encode_dna(example)

fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(encoded, aspect='auto', cmap='Blues')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['A', 'C', 'G', 'T'])
ax.set_xticks(range(len(example)))
ax.set_xticklabels(list(example))
ax.set_title('One-Hot Encoding of DNA Sequence')
ax.set_xlabel('Position')

# Add text annotations
for i in range(4):
    for j in range(len(example)):
        ax.text(j, i, int(encoded[i, j]), ha='center', va='center',
                color='white' if encoded[i, j] > 0.5 else 'gray')

plt.tight_layout()
plt.show()

print(f"One-hot shape: {encoded.shape} (4 bases x {len(example)} positions)")
print("This is the standard input format for 1D CNNs on DNA sequences.")

---
## 10. Complete Workflow: Promoter Classification Pipeline

Let us put the full ML workflow together in a clean pipeline.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning for Random Forest
param_grid = {
    'clf__n_estimators': [100, 200, 500],
    'clf__max_depth': [5, 10, 15, None],
    'clf__min_samples_leaf': [1, 3, 5],
}

rf_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    rf_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.3f}")
print(f"Test F1 score:    {f1_score(y_test, grid_search.predict(X_test)):.3f}")

In [ ]:
# Final model: visualize the decision boundary in 2D PCA space
best_model = grid_search.best_estimator_

# Project to 2D for visualization
pca_2d = PCA(n_components=2)
X_train_2d = pca_2d.fit_transform(X_train)
X_test_2d = pca_2d.transform(X_test)

# Train a simple model on 2D data for visualization
rf_2d = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_2d.fit(X_train_2d, y_train)

# Create mesh grid
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
h = 0.1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = rf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#0000FF'])

ax.pcolormesh(xx, yy, Z, cmap=cmap_light, alpha=0.3)
scatter = ax.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap=cmap_bold,
                     s=20, alpha=0.6, edgecolors='white', linewidths=0.5)
ax.scatter(X_test_2d[:, 0], X_test_2d[:, 1], c=y_test, cmap=cmap_bold,
           s=60, alpha=0.9, edgecolors='black', linewidths=1.5, marker='D')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title(f'Decision Boundary (PCA projection)\nTest accuracy: {rf_2d.score(X_test_2d, y_test):.1%}\n(circles=train, diamonds=test, red=non-promoter, blue=promoter)')
plt.tight_layout()
plt.show()

---
## 11. Summary

| Topic | Key points |
|-------|------------|
| **ML in biology** | Classification, regression, clustering -- all widely used |
| **Feature engineering** | k-mer frequencies, physicochemical properties, domain-specific features |
| **Algorithms** | Logistic regression (interpretable), Random Forest (robust), SVM (high-dimensional), k-NN (simple) |
| **Evaluation** | Accuracy, precision, recall, F1, ROC AUC; always use held-out test data |
| **Cross-validation** | k-fold CV for reliable performance estimates |
| **Unsupervised** | PCA/t-SNE for visualization, k-means for clustering |
| **Deep learning** | CNNs learn motifs from raw sequence; transformers power AlphaFold |
| **Overfitting** | The central challenge; combat with regularization, cross-validation, and proper train/test splits |

---
## Exercises

### Exercise 1: Feature Engineering -- Trinucleotide Features

Extend the promoter classification pipeline:

1. Write a function that adds **trinucleotide (k=3) frequencies** to the feature set (64 additional features)
2. Re-extract features for all 600 sequences with both dinucleotide and trinucleotide features
3. Train a Random Forest on the expanded feature set
4. Compare performance (accuracy, F1) to the dinucleotide-only model
5. Do trinucleotide features improve performance? Which trinucleotides are most important?

In [ ]:
# Your solution here


### Exercise 2: Hyperparameter Tuning

Using the promoter dataset:

1. For k-NN, test k values from 1 to 30. Plot accuracy vs. k.
2. For SVM, test different kernels ('linear', 'rbf', 'poly') and C values (0.01, 0.1, 1, 10, 100)
3. Use `GridSearchCV` with 5-fold CV for the SVM tuning
4. Report the best hyperparameters and compare the optimized SVM to the optimized RF from Section 10

In [ ]:
# Your solution here


### Exercise 3: Imbalanced Classification

In real bioinformatics, classes are often imbalanced (e.g., 5% pathogenic variants, 95% benign).

1. Create an imbalanced dataset: 50 promoters and 500 non-promoters
2. Train a Random Forest with default settings. What is the accuracy? Is it meaningful?
3. Try three strategies:
   - `class_weight='balanced'` in RandomForestClassifier
   - Oversampling the minority class (duplicate promoter sequences)
   - Undersampling the majority class (randomly select 50 non-promoters)
4. Compare using F1-score and precision-recall curves (NOT just accuracy)

In [ ]:
# Your solution here


### Exercise 4: Clustering Gene Expression Data

Simulate an expression dataset:
- 500 genes, 40 samples
- 4 cancer subtypes (10 samples each) with different expression patterns
- Subtype 1: genes 0-30 upregulated
- Subtype 2: genes 31-60 upregulated
- Subtype 3: genes 61-90 upregulated
- Subtype 4: no genes specifically upregulated (baseline)

Tasks:
1. Apply PCA and t-SNE to visualize the samples
2. Use k-means to cluster the samples. Try k=2, 3, 4, 5
3. Compute the Adjusted Rand Index (ARI) for each k vs the true subtypes
4. Which k gives the best agreement with true subtypes?

In [ ]:
# Your solution here


### Exercise 5: End-to-End Classification Pipeline

Build a complete pipeline to classify protein sequences as **enzymes** or **non-enzymes**.

1. Generate 200 "enzyme" sequences (higher proportion of catalytic residues: H, C, D, E, S) and 200 "non-enzyme" sequences (more hydrophobic: A, V, L, I, F)
2. Feature engineering: amino acid composition (20 features), dipeptide frequencies (400 features), molecular weight estimate, charge estimate
3. Train three models: Logistic Regression, Random Forest, SVM
4. Compare using 5-fold CV
5. For the best model, identify the 10 most important features
6. Create a final report with ROC curves for all three models

In [ ]:
# Your solution here


### Exercise 6: Learning Curves

Using the promoter classification dataset:

1. Train a Random Forest with increasing amounts of training data: 10%, 20%, 30%, ..., 100% of the training set
2. For each training size, compute both training accuracy and cross-validated test accuracy
3. Plot the **learning curve** (training size vs accuracy)
4. Answer: Is the model underfitting, overfitting, or well-fitted? Would more data help?
5. Repeat for Logistic Regression. How do the learning curves differ?

In [ ]:
# Your solution here


---
## 12. Random Forest with Real Data

Random Forests are ensemble methods that build multiple decision trees and aggregate predictions. They handle high-dimensional biological data well and provide feature importance rankings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the wine dataset (178 samples, 3 classes, 13 chemical features)
wine_path = "../../Assets/data/wine.csv"
feature_names = [
    "alcohol", "malic_acid", "ash", "alcalinity_ash", "magnesium",
    "total_phenols", "flavanoids", "nonflavanoid_phenols",
    "proanthocyanins", "color_intensity", "hue",
    "od280_od315", "proline"
]
wine_df = pd.read_csv(wine_path, header=None,
                      names=["class"] + feature_names)

X = wine_df[feature_names].values
y = wine_df["class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"Training samples: {X_train.shape[0]}  |  Test samples: {X_test.shape[0]}")
print(f"Train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"Test  accuracy: {rf.score(X_test,  y_test):.4f}")

In [ ]:
from sklearn.metrics import classification_report

y_pred = rf.predict(X_test)
print("Classification report (wine dataset):")
print(classification_report(y_pred=y_pred, y_true=y_test,
                             target_names=["Class 1", "Class 2", "Class 3"]))

In [ ]:
from sklearn.decomposition import PCA

# PCA: 2-D projection to visualise train / test separation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d  = pca.transform(X_test_sc)

colors = {1: "steelblue", 2: "darkorange", 3: "seagreen"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, X2d, y_arr, title in [
        (axes[0], X_train_2d, y_train, "Training data"),
        (axes[1], X_test_2d,  y_test,  "Test data")]:
    for cls in sorted(np.unique(y_arr)):
        mask = y_arr == cls
        ax.scatter(X2d[mask, 0], X2d[mask, 1],
                   c=colors[cls], label=f"Class {cls}",
                   alpha=0.8, edgecolors="white", linewidths=0.4, s=60)
    ax.set_title(title)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)")
    ax.legend()

plt.suptitle("PCA of wine dataset — Random Forest train/test split", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance bar plot
importance_df = pd.DataFrame({
    "feature":   feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance_df["feature"], importance_df["importance"],
        color="steelblue", edgecolor="white")
ax.set_xlabel("Mean decrease in impurity (Gini importance)")
ax.set_title("Random Forest — feature importances (wine dataset)")
plt.tight_layout()
plt.show()

### Random Forests in genomics

In genomics, Random Forests are used for:
- Cancer type classification from expression profiles
- Variant pathogenicity prediction (e.g. ClinVar benign vs. pathogenic)
- Drug response prediction from pharmacogenomics panels

The feature importance mechanism is especially valuable: when classifying cancer subtypes from RNA-seq data, the top-ranked genes become candidates for biomarker validation.

### Exercise: Parkinson's Disease Classification

The `parkinsons.csv` dataset contains vocal measurements collected from 195 recordings of 31 people (23 with Parkinson's Disease). The `status` column is the target label (1 = PD, 0 = healthy).

Your task:
1. Load `../../Assets/data/parkinsons.csv`; the `name` column is an identifier — drop it.
2. Split into 80 % train / 20 % test (stratified, `random_state=42`).
3. Train a `RandomForestClassifier` (200 trees).
4. Report `classification_report` and overall accuracy.
5. *(Bonus)* Plot the top-10 most important vocal features.

In [ ]:
# Your solution here


---

[← Previous: Statistics for Bioinformatics](../../Tier_3_Applied_Bioinformatics/06_Statistics_for_Bioinformatics/01_statistics_for_bioinformatics.ipynb) | [Next: From Sequence to Discovery: An Integrative Bioinformatics Project →](../../Tier_3_Applied_Bioinformatics/08_Capstone_Project/01_capstone_project.ipynb)